# Switching between Image and Video

The only difference is the `media_type` kwarg passed to `load_zoo_model`.
Everything else — prompt, `apply_model`, per-sample fields — works identically.

| `media_type` | Model class | Output |
|---|---|---|
| `"image"` (default) | `MolmoPointImageModel` | sample-level `fo.Keypoints` |
| `"video"` | `MolmoPointVideoModel` | depends on `operation` (see below) |

In [1]:
import fiftyone as fo 
import fiftyone.zoo as foz 

dataset = foz.load_zoo_dataset(
    "quickstart-video",
    overwrite=True
    )

dataset.compute_metadata()

/home/harpreet/miniconda3/envs/fo_video_workshop/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Overwriting existing directory '/home/harpreet/fiftyone/quickstart-video'
 100% |████|  281.7Mb/281.7Mb [2.0s elapsed, 0s remaining, 158.0Mb/s]      
Extracting dataset...
Parsing dataset metadata
Found 10 samples
Dataset info written to '/home/harpreet/fiftyone/quickstart-video/info.json'
Loading existing dataset 'quickstart-video'. To reload from disk, either delete the existing dataset or provide a custom `dataset_name` to use


In [2]:
 # Get unique labels across all frames for each sample
sample_objects = [dataset.distinct('frames.detections.detections.label')] * len(dataset)

dataset.set_values("sample_objects", sample_objects)


## Video — Tracking

Tracking follows objects across frames and writes frame-level `fo.Keypoints`.

Each `fo.Keypoint` has:
- `label` — the object name you prompted with
- `index` — the integer object ID assigned by the model (useful for re-identifying the same object across frames)
- `points` — normalized `[[x, y]]` coordinate

> **Note:** Call `dataset.compute_metadata()` before running tracking so the
> model can convert timestamps to accurate frame numbers. It falls back to
> 30 fps with a warning if metadata is missing.

In [3]:
import fiftyone.zoo as foz

foz.register_zoo_model_source(
    "https://github.com/harpreetsahota204/molmo_point", 
    overwrite=True
    )


model = foz.load_zoo_model(
    "allenai/MolmoPoint-8B",
    media_type="video",
)

model.operation="tracking"  # frame-level keypoints, samples at max_fps=10

  274.5Mb [1.9s elapsed, ? remaining, 141.3Mb/s] 
Overwriting existing model source '/home/harpreet/fiftyone/__models__/molmo-point'


Fetching 33 files: 100%|██████████| 33/33 [00:03<00:00,  8.93it/s]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 8/8 [00:04<00:00,  1.78it/s]


In [4]:
dataset.apply_model(
    model,
    prompt_field="sample_objects",
    label_field="tracking_keypoints",  # written to sample.frames[n].keypoints
    batch_size=4,             # videos are processed sequentially; batch_size controls DataLoader prefetch
    num_workers=2,
    skip_failures=False
)

   0% ||------------------|  0/10 [2.8ms elapsed, ? remaining, ? samples/s] 

molmo-utils using decord by default to read video.


 100% |███████████████████| 10/10 [13.8m elapsed, 0s remaining, 0.0 samples/s]    


## Video — Pointing

Pointing samples the video sparsely (`max_fps=2`) and writes frame-level `fo.Keypoints` on the frames where detections were found — useful when you just want to confirm that an object is present somewhere in the video without dense per-frame tracking.

Switching `operation` automatically updates `max_fps` to the new default (2 for pointing) unless you've explicitly pinned it.

In [5]:
model.operation = "pointing"
model.prompt = ["parked car", "restaurant", "pedestrian", "house"]

dataset.apply_model(
    model,
    label_field="static_keypoints",
    batch_size=4,
    num_workers=2,
    skip_failures=False
)

 100% |███████████████████| 10/10 [3.2m elapsed, 0s remaining, 0.0 samples/s]     


## Tuning inference parameters

All parameters can be set at load time or changed on the model after loading — the model stays on the GPU, no reload needed.

| Parameter | Default | Notes |
|---|---|---|
| `operation` | `"tracking"` | Switching auto-updates `max_fps` unless it was pinned |
| `max_fps` | 10 (tracking) / 2 (pointing) | Frames per second the model sees |
| `num_frames` | `None` (processor default, 384) | Hard cap on total frames passed to the model |
| `frame_sample_mode` | `None` (processor default) | `"fps"` or `"uniform_last_frame"` |
| `interp_max_gap` | `None` → 1 second of frames | Max gap (frames) to bridge with linear interpolation in tracking |

The relationship between `max_fps` and `interp_max_gap`: higher `max_fps` produces denser keyframes so less interpolation is needed. Lower `max_fps` produces sparser keyframes and more frames get filled by interpolation. `interp_max_gap` is the safety net that prevents interpolation from silently bridging gaps where the object genuinely left the scene.

In [ ]:
# All parameters can also be set at load time
model = foz.load_zoo_model(
    "allenai/MolmoPoint-8B",
    media_type="video",
    operation="tracking",
    max_fps=10,
    num_frames=128,           # cap at 128 frames per video
    frame_sample_mode="fps",  # sample at max_fps intervals
    interp_max_gap=60,        # bridge gaps up to 60 frames (~2 sec at 30fps)
)

# Or tweak on an already-loaded model without reloading weights
model.num_frames = 128
model.frame_sample_mode = "fps"
model.interp_max_gap = 60

# max_fps is coupled to operation — switching operation auto-updates max_fps
# unless you've explicitly set it
model.operation = "pointing"  # max_fps automatically becomes 2
print(f"operation={model.operation}, max_fps={model.max_fps}")

model.max_fps = 5             # pin it — won't change when switching operation
model.operation = "tracking"  # max_fps stays 5
print(f"operation={model.operation}, max_fps={model.max_fps}")

model.max_fps = None          # reset to automatic default
print(f"operation={model.operation}, max_fps={model.max_fps}")